### Stationarity & Order of Integration
**Goal:** find d (order of integration) for every series → feeds ARIMA(p,d,q).

**Why it matters:** fitting ARIMA/VAR on non-stationary data gives spurious regression (fake significant results from shared trends)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings("ignore", message="The test statistic is outside")

In [2]:
panel = pd.read_parquet(r"D:\yield_curve_forecast\data\processed\yield_curve_data.parquet").set_index("observation_date")

yield_col = ["DGS3MO", "DGS1", "DGS2", "DGS5", "DGS10", "DGS30"]
macro_col = [c for c in panel.columns if c not in yield_col]

print("panel shape:", panel.shape)
print("yields :", yield_col)
print("macro  :", macro_col)
panel.head()

panel shape: (405, 18)
yields : ['DGS3MO', 'DGS1', 'DGS2', 'DGS5', 'DGS10', 'DGS30']
macro  : ['CPIAUCSL', 'CPILFESL', 'DCOILWTICO', 'FEDFUNDS', 'GDPC1', 'HOUST', 'ICSA', 'INDPRO', 'RSAFS', 'UMCSENT', 'UNRATE', 'VIXCLS']


,CPIAUCSL,CPILFESL,DCOILWTICO,DGS1,DGS10,DGS2,DGS30,DGS3MO,DGS5,FEDFUNDS,GDPC1,HOUST,ICSA,INDPRO,RSAFS,UMCSENT,UNRATE,VIXCLS
observation_date,,,,,,,,,,,,,,,,,,
1992-01-01,138.3,145.1,18.93,4.23,7.31,5.11,7.77,3.94,6.44,4.03,10236.435,1176,439000.0,61.4616,159177,67.5,7.3,17.40
1992-02-01,138.6,145.4,18.69,4.35,7.27,5.27,7.80,4.03,6.58,4.06,10236.435,1250,442200.0,61.8955,159189,68.8,7.4,16.68
1992-03-01,139.1,145.9,19.49,4.54,7.54,5.60,7.96,4.15,6.94,3.98,10236.435,1297,429500.0,62.4232,158647,76.0,7.4,16.18
1992-04-01,139.4,146.3,20.88,4.40,7.61,5.46,8.06,3.79,6.91,3.73,10347.429,1099,418250.0,62.8970,159921,77.2,7.4,15.53
1992-05-01,139.7,146.8,22.13,4.24,7.33,5.19,7.84,3.79,6.61,3.82,10347.429,1214,417400.0,63.1040,160471,79.2,7.6,13.86


In [3]:
def adf_pvalue(series, regression="c"):
    return adfuller(series.dropna(), regression=regression, autolag="AIC")[1]

def kpss_pvalue(series, regression="c"):
    return kpss(series.dropna(), regression=regression, nlags="auto")[1]

def stationarity_verdict(adf_p, kpss_p, alpha=0.05):
    adf_stat  = adf_p  < alpha     # rejected unit root
    kpss_stat = kpss_p >= alpha    # failed to reject stationarity
    if adf_stat and kpss_stat:         return "Stationary"
    if (not adf_stat) and (not kpss_stat): return "Unit root"
    if adf_stat and not kpss_stat:     return "Trend-stationary?"
    return "Inconclusive"

In [4]:
rows = []
for col in panel.columns:
    ap, kp = adf_pvalue(panel[col]), kpss_pvalue(panel[col])
    rows.append({"series": col, "ADF_p": ap, "KPSS_p": kp,
                 "verdict": stationarity_verdict(ap, kp)})

level_tests = pd.DataFrame(rows).set_index("series")
level_tests

,ADF_p,KPSS_p,verdict
series,,,
CPIAUCSL,0.997769,0.010000,Unit root
CPILFESL,0.998712,0.010000,Unit root
DCOILWTICO,0.124407,0.010000,Unit root
DGS1,0.049250,0.010000,Trend-stationary?
DGS10,0.173605,0.010000,Unit root
DGS2,0.166053,0.010000,Unit root
DGS30,0.157682,0.010000,Unit root
DGS3MO,0.042965,0.010000,Trend-stationary?
DGS5,0.268618,0.010000,Unit root


In [5]:
def integration_order(series, max_d=2, alpha=0.05):
    s = series.dropna().copy()
    for d in range(max_d + 1):
        ap, kp = adf_pvalue(s), kpss_pvalue(s)
        if stationarity_verdict(ap, kp, alpha) == "Stationary":
            return d, ap, kp
        s = s.diff().dropna()
    return max_d, ap, kp   # not stationary even at max_d

transform_map = {0: "none (level is I(0))", 1: "first difference", 2: "second difference"}

rows = []
for col in panel.columns:
    d, ap, kp = integration_order(panel[col])
    rows.append({"series": col, "integration_order_d": d,
                 "ADF_p_at_d": ap, "KPSS_p_at_d": kp,
                 "recommended_transform": transform_map[d]})

integ = (pd.DataFrame(rows).set_index("series")
         .sort_values("integration_order_d"))
integ

,integration_order_d,ADF_p_at_d,KPSS_p_at_d,recommended_transform
series,,,,
ICSA,0,1.002173e-06,0.100000,none (level is I(0))
UNRATE,0,3.091802e-02,0.100000,none (level is I(0))
VIXCLS,0,5.287944e-03,0.100000,none (level is I(0))
DGS1,1,6.464478e-05,0.100000,first difference
DGS30,1,2.345641e-20,0.100000,first difference
DGS3MO,1,7.464484e-04,0.100000,first difference
DGS2,1,2.198704e-21,0.100000,first difference
DCOILWTICO,1,2.142405e-29,0.100000,first difference
HOUST,1,4.043476e-28,0.100000,first difference


In [6]:
integ.to_csv(r"D:\yield_curve_forecast\data\processed\integration_orders.csv")
print("Saved -> integration_orders.csv\n")
print("Order-of-integration counts:")
print(integ["integration_order_d"].value_counts().sort_index().to_string())

Saved -> integration_orders.csv

Order-of-integration counts:
integration_order_d
0     3
1    13
2     2


In [7]:
feat = pd.read_parquet(r"D:\yield_curve_forecast\data\processed\features.parquet")
if "observation_date" in feat.columns:
    feat = feat.set_index("observation_date")   

engineered = [
    "INFLATION_YOY", "CORE_INFLATION_YOY", "GDP_GROWTH_YOY",
    "INDPRO_GROWTH_YOY", "RSAFS_GROWTH_YOY", "HOUST_GROWTH_YOY",
    "SPREAD_10Y_2Y", "SPREAD_10Y_3M", "SPREAD_2Y_3M",
    "CURVE_LEVEL", "CURVE_SLOPE", "CURVE_CURVATURE",
    "VIX_LOG", "WTI_LOG_RETURN",
    "DGS10_CHANGE", "FEDFUNDS_CHANGE", "UNRATE_CHANGE",
]
engineered = [c for c in engineered if c in feat.columns]

rows = []
for col in engineered:
    ap, kp = adf_pvalue(feat[col]), kpss_pvalue(feat[col])
    rows.append({"feature": col, "ADF_p": ap, "KPSS_p": kp,
                 "verdict": stationarity_verdict(ap, kp)})

feature_stationarity = pd.DataFrame(rows).set_index("feature")
feature_stationarity

,ADF_p,KPSS_p,verdict
feature,,,
INFLATION_YOY,8.762012e-03,0.100000,Stationary
CORE_INFLATION_YOY,1.127978e-01,0.027369,Unit root
GDP_GROWTH_YOY,1.418175e-02,0.090143,Stationary
INDPRO_GROWTH_YOY,1.088237e-02,0.029938,Trend-stationary?
RSAFS_GROWTH_YOY,2.542907e-03,0.100000,Stationary
HOUST_GROWTH_YOY,1.129704e-01,0.100000,Inconclusive
SPREAD_10Y_2Y,3.476346e-02,0.076313,Stationary
SPREAD_10Y_3M,1.590289e-02,0.035578,Trend-stationary?
SPREAD_2Y_3M,4.804339e-03,0.010000,Trend-stationary?
